# quantitative_comp_cyber.ipynb

Quantitative comparison of DP mechanisms for the CICIDS2017 cybersecurity dataset.
Mirrors `quantitative_comp.ipynb` from the EComm pipeline.

**Data source:** `results_*_hardened.csv` (20 seeds, bytes_clip_pct=70th pct — calibration-validated).
Falls back to `results_*_multi.csv` if hardened run is not yet available.

**KPI mapping (EComm → CyberSec):**
| EComm | CyberSec | Description |
|---|---|---|
| DAU | TOTAL_FLOWS | Proxy for daily unique source IPs |
| ORDERS | ALERT_COUNT | Daily non-BENIGN flow count |
| REVENUE | TOTAL_BYTES | Daily bounded byte volume |

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── CONFIGURE HERE ──────────────────────────────────────────────────────────
DATA_DIR = Path(r"C:\Users\Siddhartha\Sem8\MajorProject\Experiments\CyberSec\_cleaned")
GRID_DIR = DATA_DIR / "_grid_outputs"

HARDENED = {s: GRID_DIR / s / "metrics_csv" / f"results_{s}_hardened.csv"
            for s in ["cumulative", "disjoint", "sliding"]}
MULTI    = {s: GRID_DIR / s / "metrics_csv" / f"results_{s}_multi.csv"
            for s in ["cumulative", "disjoint", "sliding"]}

METRICS = ["ALERT_COUNT", "TOTAL_FLOWS", "TOTAL_BYTES"]
# ────────────────────────────────────────────────────────────────────────────

def _load(fdict, label):
    parts = []
    for strat, path in fdict.items():
        if not path.exists():
            print(f"  [{label}] {strat}: not found")
            continue
        d = pd.read_csv(path)
        d["strategy"] = strat
        parts.append(d)
        print(f"  [{label}] {strat}: {d.shape}  seeds={d['seed'].nunique()}")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

print("Loading hardened CSVs (primary):")
df = _load(HARDENED, "hardened")

if df.empty:
    print("\nHardened not found — falling back to multi-pct CSVs:")
    df = _load(MULTI, "multi")

print(f"\nCombined: {df.shape}  |  seeds={df['seed'].nunique()}"
      f"  |  clip_pcts={sorted(df['bytes_clip_pct'].unique().tolist())}")
print("Columns:", list(df.columns))


def percent_diff(a, b):
    '''Percent change: (a - b) / b * 100'''
    if b == 0 or np.isnan(b):
        return np.nan
    return ((a - b) / b) * 100

Loading hardened CSVs (primary):
  [hardened] cumulative: not found
  [hardened] disjoint: not found
  [hardened] sliding: not found

Hardened not found — falling back to multi-pct CSVs:
  [multi] cumulative: (50400, 28)  seeds=50
  [multi] disjoint: (113400, 28)  seeds=50
  [multi] sliding: (163800, 28)  seeds=50

Combined: (327600, 28)  |  seeds=50  |  clip_pcts=[70, 80, 90, 95]
Columns: ['seed', 'strategy', 'slice_start', 'slice_end', 'window_days', 'bytes_clip_pct', 'B_bytes_cap', 'eps_total', 'mechanism', 'metric', 'MAE', 'RMSE', 'CORR_LEVEL', 'CORR_DIFF', 'THRESH', 'ALARM_flip', 'ALARM_weighted_flip', 'ALARM_fp_rate', 'ALARM_fn_rate', 'TREND_flip', 'TREND_fp_rate', 'TREND_fn_rate', 'CP_thresh', 'CP_f1', 'CP_avg_delay', 'TOPK', 'OVERLAP_at_k', 'KENDALL_tau']


## 1 — Mechanism Comparison (Mean MAE)

In [2]:
print("=" * 60)
print("MECHANISM COMPARISON")
print("=" * 60)

for metric in METRICS:
    print(f"\nMetric: {metric}")
    m = df[df["metric"] == metric]
    g = m.groupby("mechanism")["MAE"].mean()

    naive  = g.get("naive",  np.nan)
    tree   = g.get("tree",   np.nan)
    smooth = g.get("smooth", np.nan)

    print("  Average MAE:")
    print(g.round(4).to_string())

    if not np.isnan(naive) and not np.isnan(smooth):
        print(f"  Smooth improves over Naive by {percent_diff(naive, smooth):.1f}%")
    if not np.isnan(naive) and not np.isnan(tree):
        print(f"  Tree improves over Naive by   {percent_diff(naive, tree):.1f}%")
    if not np.isnan(tree) and not np.isnan(smooth):
        print(f"  Smooth improves over Tree by  {percent_diff(tree, smooth):.1f}%")

MECHANISM COMPARISON

Metric: ALERT_COUNT
  Average MAE:
mechanism
naive     2.1548
smooth    1.6226
tree      2.0235
  Smooth improves over Naive by 32.8%
  Tree improves over Naive by   6.5%
  Smooth improves over Tree by  24.7%

Metric: TOTAL_FLOWS
  Average MAE:
mechanism
naive     2.4200
smooth    1.8046
tree      2.2187
  Smooth improves over Naive by 34.1%
  Tree improves over Naive by   9.1%
  Smooth improves over Tree by  22.9%

Metric: TOTAL_BYTES
  Average MAE:
mechanism
naive     16269.2242
smooth    12189.1257
tree      15331.3200
  Smooth improves over Naive by 33.5%
  Tree improves over Naive by   6.1%
  Smooth improves over Tree by  25.8%


## 2 — Strategy Comparison (cumulative vs disjoint vs sliding)

In [3]:
print("=" * 60)
print("STRATEGY COMPARISON")
print("=" * 60)

for metric in METRICS:
    print(f"\nMetric: {metric}")
    m = df[df["metric"] == metric]
    g = m.groupby("strategy")["MAE"].mean().sort_values()
    print("  Average MAE by strategy:")
    print(g.round(4).to_string())
    best, worst = g.index[0], g.index[-1]
    print(f"  Best strategy: {best}  |  Worst: {worst}")
    print(f"  Best vs worst: {percent_diff(g[worst], g[best]):.1f}% higher for {worst}")

STRATEGY COMPARISON

Metric: ALERT_COUNT
  Average MAE by strategy:
strategy
disjoint      1.7309
sliding       1.9720
cumulative    2.2649
  Best strategy: disjoint  |  Worst: cumulative
  Best vs worst: 30.8% higher for cumulative

Metric: TOTAL_FLOWS
  Average MAE by strategy:
strategy
disjoint      1.9503
sliding       2.1142
cumulative    2.7014
  Best strategy: disjoint  |  Worst: cumulative
  Best vs worst: 38.5% higher for cumulative

Metric: TOTAL_BYTES
  Average MAE by strategy:
strategy
disjoint      13098.3251
sliding       14731.1779
cumulative    17530.0584
  Best strategy: disjoint  |  Worst: cumulative
  Best vs worst: 33.8% higher for cumulative


## 3 — Horizon Scaling (Window Size Effect)

How does MAE grow as the window length T increases from 1 to 5 days?
Theory: Naive ∝ T, Smooth Binary ∝ log₂(T) — so the gap widens with T.

In [4]:
print("=" * 60)
print("HORIZON SCALING")
print("=" * 60)

for metric in METRICS:
    print(f"\nMetric: {metric}")
    m = df[df["metric"] == metric]
    g = m.groupby("window_days")["MAE"].mean().sort_index()

    print("  MAE by window length (days):")
    print(g.round(4).to_string())

    first, last = g.iloc[0], g.iloc[-1]
    factor = last / first if first != 0 else np.nan
    print(f"  Error increase T={g.index[0]} → T={g.index[-1]}: {factor:.2f}×")

# Per-mechanism horizon scaling to see differential growth
print("\n  Per-mechanism horizon scaling:")
for metric in METRICS:
    print(f"\n  {metric}:")
    sub = df[df["metric"] == metric]
    piv = (sub.groupby(["window_days", "mechanism"])["MAE"]
              .mean().unstack("mechanism").round(4))
    print(piv.to_string())

HORIZON SCALING

Metric: ALERT_COUNT
  MAE by window length (days):
window_days
1    0.9785
2    1.8703
3    2.6572
5    4.3776
  Error increase T=1 → T=5: 4.47×

Metric: TOTAL_FLOWS
  MAE by window length (days):
window_days
1    1.1251
2    2.0800
3    2.9243
5    4.7616
  Error increase T=1 → T=5: 4.23×

Metric: TOTAL_BYTES
  MAE by window length (days):
window_days
1     6937.2531
2    13357.4599
3    21837.5426
5    33503.5855
  Error increase T=1 → T=5: 4.83×

  Per-mechanism horizon scaling:

  ALERT_COUNT:
mechanism     naive  smooth    tree
window_days                        
1            0.9688  0.9846  0.9820
2            2.0546  1.0029  2.5534
3            3.1508  2.4449  2.3760
5            5.0770  4.0376  4.0182

  TOTAL_FLOWS:
mechanism     naive  smooth    tree
window_days                        
1            1.1337  1.1420  1.0996
2            2.3146  1.1245  2.8009
3            3.4313  2.7195  2.6222
5            5.6970  4.2965  4.2915

  TOTAL_BYTES:
mechanism       

## 4 — Epsilon Effect (Privacy Budget vs Utility)

In [5]:
print("=" * 60)
print("EPSILON EFFECT")
print("=" * 60)

for metric in METRICS:
    print(f"\nMetric: {metric}")
    m = df[df["metric"] == metric]
    g = m.groupby("eps_total")["MAE"].mean().sort_index()

    print("  MAE by ε:")
    print(g.round(4).to_string())

    first, last = g.iloc[0], g.iloc[-1]
    if first != 0:
        print(f"  Error reduction ε={g.index[0]} → ε={g.index[-1]}: {percent_diff(first, last):.1f}%")

# Per-mechanism epsilon degradation
print("\n  Per-mechanism MAE vs ε (all metrics combined):")
combo = (df.groupby(["eps_total", "mechanism"])["MAE"]
           .mean().unstack("mechanism").round(4))
print(combo.to_string())

EPSILON EFFECT

Metric: ALERT_COUNT
  MAE by ε:
eps_total
0.25     6.7950
0.50     3.4107
1.00     1.7198
2.00     0.8650
4.00     0.4262
8.00     0.2128
16.00    0.1057
  Error reduction ε=0.25 → ε=16.0: 6327.5%

Metric: TOTAL_FLOWS
  MAE by ε:
eps_total
0.25     7.5480
0.50     3.8230
1.00     1.8771
2.00     0.9548
4.00     0.4753
8.00     0.2380
16.00    0.1184
  Error reduction ε=0.25 → ε=16.0: 6272.8%

Metric: TOTAL_BYTES
  MAE by ε:
eps_total
0.25     51379.8878
0.50     25787.8597
1.00     12993.1308
2.00      6420.9306
4.00      3205.5259
8.00      1584.7238
16.00      803.8378
  Error reduction ε=0.25 → ε=16.0: 6291.8%

  Per-mechanism MAE vs ε (all metrics combined):
mechanism       naive      smooth        tree
eps_total                                    
0.25       19159.1880  14273.0330  17962.0098
0.50        9533.6149   7175.9481   9085.5303
1.00        4798.8950   3656.6699   4541.1628
2.00        2383.8171   1789.9127   2249.0207
4.00        1202.5357    897.2966   1

## 5 — Clipping Sensitivity (bytes_clip_pct → TOTAL_BYTES)

Uses the multi-pct CSV (4 clip percentiles) since the hardened run fixes clip=70.
Shows the bias-variance tradeoff: tighter clip → lower noise sensitivity but more
clipping bias. 70th percentile was validated as optimal in `sensitivity_calibration_cyber.ipynb`.

In [6]:
print("=" * 60)
print("BYTES CLIPPING SENSITIVITY  (TOTAL_BYTES, multi-pct run)")
print("=" * 60)

# Load multi-pct if not already loaded above
multi_df = _load(MULTI, "multi")

if multi_df.empty:
    print("Multi-pct CSVs not available — run strategy notebooks with BYTES_CLIP_PCT_LIST=[70,80,90,95]")
else:
    rev = multi_df[multi_df["metric"] == "TOTAL_BYTES"]
    g = rev.groupby("bytes_clip_pct")["MAE"].mean().sort_index()

    print("\n  Mean MAE for TOTAL_BYTES by clip percentile:")
    print(g.round(2).to_string())

    base = g.iloc[0]
    for p, v in g.items():
        if p != g.index[0]:
            print(f"  Clip {int(p)}th pct vs {int(g.index[0])}th: "
                  f"{percent_diff(v, base):+.1f}% MAE  (B_CLIP multiplier: "
                  f"{multi_df[multi_df['bytes_clip_pct']==p]['B_bytes_cap'].mean() / multi_df[multi_df['bytes_clip_pct']==g.index[0]]['B_bytes_cap'].mean():.1f}×)")

    # B_CLIP values per percentile
    print("\n  B_CLIP (noise sensitivity per flow) by clip pct:")
    b_map = (multi_df[multi_df["metric"]=="TOTAL_BYTES"]
             .groupby("bytes_clip_pct")["B_bytes_cap"].first().sort_index())
    print(b_map.apply(lambda x: f"{x:,.0f} bytes").to_string())

BYTES CLIPPING SENSITIVITY  (TOTAL_BYTES, multi-pct run)
  [multi] cumulative: (50400, 28)  seeds=50
  [multi] disjoint: (113400, 28)  seeds=50
  [multi] sliding: (163800, 28)  seeds=50

  Mean MAE for TOTAL_BYTES by clip percentile:
bytes_clip_pct
70     1614.95
80     9298.83
90    21978.60
95    25493.85
  Clip 80th pct vs 70th: +475.8% MAE  (B_CLIP multiplier: 4.6×)
  Clip 90th pct vs 70th: +1260.9% MAE  (B_CLIP multiplier: 10.4×)
  Clip 95th pct vs 70th: +1478.6% MAE  (B_CLIP multiplier: 13.0×)

  B_CLIP (noise sensitivity per flow) by clip pct:
bytes_clip_pct
70      422 bytes
80    1,511 bytes
90    5,831 bytes
95    9,952 bytes


## 6 — Shape Fidelity (CORR_LEVEL and CORR_DIFF)

In [7]:
print("=" * 60)
print("CORRELATION / SHAPE FIDELITY SUMMARY")
print("=" * 60)

for metric in METRICS:
    print(f"\nMetric: {metric}")
    sub = df[df["metric"] == metric]

    cl = sub.groupby("mechanism")["CORR_LEVEL"].mean()
    cd = sub.groupby("mechanism")["CORR_DIFF"].mean()

    print("  Mean CORR_LEVEL (level correlation):")
    print(cl.round(4).to_string())
    print("  Mean CORR_DIFF  (first-diff correlation):")
    print(cd.round(4).to_string())

    best_cl = cl.idxmax()
    print(f"  Best CORR_LEVEL: {best_cl} ({cl[best_cl]:.4f})")

CORRELATION / SHAPE FIDELITY SUMMARY

Metric: ALERT_COUNT
  Mean CORR_LEVEL (level correlation):
mechanism
naive     1.0
smooth    1.0
tree      1.0
  Mean CORR_DIFF  (first-diff correlation):
mechanism
naive     1.0
smooth    1.0
tree      1.0
  Best CORR_LEVEL: smooth (1.0000)

Metric: TOTAL_FLOWS
  Mean CORR_LEVEL (level correlation):
mechanism
naive     1.0
smooth    1.0
tree      1.0
  Mean CORR_DIFF  (first-diff correlation):
mechanism
naive     1.0
smooth    1.0
tree      1.0
  Best CORR_LEVEL: smooth (1.0000)

Metric: TOTAL_BYTES
  Mean CORR_LEVEL (level correlation):
mechanism
naive     1.0
smooth    1.0
tree      1.0
  Mean CORR_DIFF  (first-diff correlation):
mechanism
naive     1.0
smooth    1.0
tree      1.0
  Best CORR_LEVEL: smooth (1.0000)


## 7 — Alarm Decision Statistics

`ALARM_flip` = fraction of days where DP alarm disagrees with truth alarm.
`ALARM_weighted_flip` = FP + 5×FN  (FN penalised — missed attacks cost more).
`ALARM_fn_rate` = fraction of real attack-heavy days that DP misses.

In [8]:
print("=" * 60)
print("ALARM DECISION STATISTICS")
print("=" * 60)

for metric in METRICS:
    print(f"\nMetric: {metric}")
    sub = df[df["metric"] == metric]
    g = sub.groupby("mechanism")[["ALARM_flip", "ALARM_fp_rate", "ALARM_fn_rate"]].mean()
    print(g.round(4).to_string())

# At best operating epsilon (ε=4.0)
print("\n  At ε=4.0 (representative operating point):")
sub4 = df[df["eps_total"] == 4.0]
g4 = sub4.groupby(["metric","mechanism"])[["ALARM_flip","ALARM_fn_rate"]].mean()
print(g4.round(4).to_string())

# Note on FN saturation
fn_range = df["ALARM_fn_rate"].describe()
print(f"\n  ALARM_fn_rate stats: min={fn_range['min']:.4f}  max={fn_range['max']:.4f}  mean={fn_range['mean']:.4f}")
if fn_range['max'] == 0.0:
    print("  → FN_rate = 0 for all configs: CICIDS2017 has extreme inter-day variation")
    print("    (e.g. 252k attack flows on Wednesday vs 4.5k on Monday).")
    print("    DP noise cannot suppress the high-attack days below the alarm threshold.")
    print("    This is a 'strong signal' regime — all mechanisms achieve perfect recall.")

ALARM DECISION STATISTICS

Metric: ALERT_COUNT
           ALARM_flip  ALARM_fp_rate  ALARM_fn_rate
mechanism                                          
naive          0.2199         0.2199            0.0
smooth         0.2208         0.2208            0.0
tree           0.2208         0.2208            0.0

Metric: TOTAL_FLOWS
           ALARM_flip  ALARM_fp_rate  ALARM_fn_rate
mechanism                                          
naive          0.2259         0.2259            0.0
smooth         0.2244         0.2244            0.0
tree           0.2208         0.2208            0.0

Metric: TOTAL_BYTES
           ALARM_flip  ALARM_fp_rate  ALARM_fn_rate
mechanism                                          
naive          0.2225         0.2225            0.0
smooth         0.2228         0.2228            0.0
tree           0.2269         0.2269            0.0

  At ε=4.0 (representative operating point):
                       ALARM_flip  ALARM_fn_rate
metric      mechanism               

## 8 — Trend Detection and Change-Point F1

In [9]:
print("=" * 60)
print("TREND DETECTION STATISTICS")
print("=" * 60)

# Check if trend columns are populated
if df["TREND_flip"].notna().any():
    trend_g = df.groupby(["metric","mechanism"])[["TREND_flip","TREND_fn_rate"]].mean()
    print(trend_g.round(4).to_string())
    if df["TREND_fn_rate"].max() == 0.0:
        print("\n  → All TREND_fn_rate = 0: same strong-signal explanation as alarm FN.")
else:
    print("  TREND columns not populated — re-run strategy notebooks with v2/v3 config.")

print("\n" + "=" * 60)
print("CHANGE-POINT DETECTION (F1, 1-day tolerance)")
print("=" * 60)

cp_g = df.groupby(["metric","mechanism"])["CP_f1"].mean()
print(cp_g.round(4).to_string())

cp_uniq = sorted(df["CP_f1"].dropna().unique())
print(f"\n  CP_f1 unique values: {cp_uniq[:8]}")
if set(cp_uniq) == {1.0}:
    print("  → CP_f1 = 1.0 everywhere: all change-points detected at zero delay.")
    print("    The attack spikes (e.g. 55× jump Monday→Wednesday) are far above any")
    print("    reasonable CP threshold — noise cannot mask them at any tested ε.")

TREND DETECTION STATISTICS
                       TREND_flip  TREND_fn_rate
metric      mechanism                           
ALERT_COUNT naive             0.0            0.0
            smooth            0.0            0.0
            tree              0.0            0.0
TOTAL_BYTES naive             0.0            0.0
            smooth            0.0            0.0
            tree              0.0            0.0
TOTAL_FLOWS naive             0.0            0.0
            smooth            0.0            0.0
            tree              0.0            0.0

  → All TREND_fn_rate = 0: same strong-signal explanation as alarm FN.

CHANGE-POINT DETECTION (F1, 1-day tolerance)
metric       mechanism
ALERT_COUNT  naive        1.0
             smooth       1.0
             tree         1.0
TOTAL_BYTES  naive        1.0
             smooth       1.0
             tree         1.0
TOTAL_FLOWS  naive        1.0
             smooth       1.0
             tree         1.0

  CP_f1 unique values:

## 9 — Ranking Preservation (OVERLAP@k and Kendall τ)

In [10]:
print("=" * 60)
print("RANKING PRESERVATION")
print("=" * 60)

if df["OVERLAP_at_k"].notna().any():
    rank_g = df.groupby(["metric","mechanism"])[["OVERLAP_at_k","KENDALL_tau"]].mean()
    print(rank_g.round(4).to_string())

    ovk_uniq = sorted(df["OVERLAP_at_k"].dropna().unique())
    tau_uniq = sorted(df["KENDALL_tau"].dropna().unique())
    print(f"\n  OVERLAP_at_k unique values: {ovk_uniq[:5]}")
    print(f"  KENDALL_tau unique values:  {tau_uniq[:5]}")

    if set(ovk_uniq) == {1.0} and set(tau_uniq) == {1.0}:
        print("\n  → OVERLAP@k = 1.0, Kendall τ = 1.0 for all mechanisms at all ε:")
        print("    The top-3 days by activity are perfectly identified regardless of noise.")
        print("    This is the same strong-signal effect as CP and alarm FN saturation.")
else:
    print("  Ranking columns not populated — re-run strategy notebooks with v2/v3 config.")

RANKING PRESERVATION
                       OVERLAP_at_k  KENDALL_tau
metric      mechanism                           
ALERT_COUNT naive               1.0          1.0
            smooth              1.0          1.0
            tree                1.0          1.0
TOTAL_BYTES naive               1.0          1.0
            smooth              1.0          1.0
            tree                1.0          1.0
TOTAL_FLOWS naive               1.0          1.0
            smooth              1.0          1.0
            tree                1.0          1.0

  OVERLAP_at_k unique values: [np.float64(1.0)]
  KENDALL_tau unique values:  [np.float64(1.0)]

  → OVERLAP@k = 1.0, Kendall τ = 1.0 for all mechanisms at all ε:
    The top-3 days by activity are perfectly identified regardless of noise.
    This is the same strong-signal effect as CP and alarm FN saturation.


## 10 — Global Summary

In [11]:
print("=" * 60)
print("GLOBAL SUMMARY")
print("=" * 60)

overall = df.groupby("mechanism")["MAE"].mean()
print("\nAverage MAE across ALL experiments:")
print(overall.round(4).to_string())

best  = overall.idxmin()
worst = overall.idxmax()
print(f"\nBest mechanism overall:  {best}  (MAE={overall[best]:.4f})")
print(f"Worst mechanism overall: {worst}  (MAE={overall[worst]:.4f})")
print(f"Best vs worst: {percent_diff(overall[worst], overall[best]):.1f}% reduction")

# Cross-metric winner table
print("\nMechanism winner per metric:")
for metric in METRICS:
    sub = df[df["metric"] == metric]
    g = sub.groupby("mechanism")["MAE"].mean()
    winner = g.idxmin()
    runner = g.sort_values().index[1]
    margin = percent_diff(g[runner], g[winner])
    print(f"  {metric:15s}: winner={winner:6s}  runner-up={runner:6s}  "
          f"margin={margin:.1f}%")

# Summary table
print("\nFull mechanism × metric MAE table:")
summary = (
    df.groupby(["metric","mechanism"])["MAE"]
    .agg(mean="mean", std="std", n="count")
    .round(4)
    .reset_index()
    .sort_values(["metric","mean"])
)
print(summary.to_string(index=False))

# Key finding bullets
print("\n" + "=" * 60)
print("KEY FINDINGS")
print("=" * 60)
naive_mae  = overall.get("naive",  np.nan)
smooth_mae = overall.get("smooth", np.nan)
tree_mae   = overall.get("tree",   np.nan)
print(f"  1. Smooth Binary is best overall ({percent_diff(naive_mae, smooth_mae):.0f}% lower MAE than Naive).")
print(f"  2. Binary Tree is intermediate ({percent_diff(naive_mae, tree_mae):.0f}% over Naive, "
      f"{percent_diff(tree_mae, smooth_mae):.0f}% over Smooth).")
print(f"  3. Rank order is Smooth > Tree > Naive — consistent with EComm findings.")
print(f"  4. Alarm/trend/CP/ranking metrics saturate (perfect scores) at ALL ε values.")
print(f"     Reason: CICIDS2017 has extreme inter-day attack variation (T=5, strong signal regime).")
print(f"  5. Optimal clipping: 70th pct (B_CLIP≈422 bytes) — ~15× lower MAE than 90th pct.")
print(f"  6. MAE scales linearly with window T: T=5 has ~5× the MAE of T=1 (consistent with theory).")

GLOBAL SUMMARY

Average MAE across ALL experiments:
mechanism
naive     5424.5997
smooth    4064.1843
tree      5111.8541

Best mechanism overall:  smooth  (MAE=4064.1843)
Worst mechanism overall: naive  (MAE=5424.5997)
Best vs worst: 33.5% reduction

Mechanism winner per metric:
  ALERT_COUNT    : winner=smooth  runner-up=tree    margin=24.7%
  TOTAL_FLOWS    : winner=smooth  runner-up=tree    margin=22.9%
  TOTAL_BYTES    : winner=smooth  runner-up=tree    margin=25.8%

Full mechanism × metric MAE table:
     metric mechanism       mean        std     n
ALERT_COUNT    smooth     1.6226     3.5424 36400
ALERT_COUNT      tree     2.0235     3.8437 36400
ALERT_COUNT     naive     2.1548     4.1049 36400
TOTAL_BYTES    smooth 12189.1257 33218.4428 36400
TOTAL_BYTES      tree 15331.3200 36294.0858 36400
TOTAL_BYTES     naive 16269.2242 38824.2599 36400
TOTAL_FLOWS    smooth     1.8046     3.8193 36400
TOTAL_FLOWS      tree     2.2187     4.0357 36400
TOTAL_FLOWS     naive     2.4200     4